# Data Preparation Pipeline

This notebook consolidates the entire data processing workflow, from cleaning the raw dataset to normalizing it into relational tables (`courses`, `categories`, `tags`, `users`, `course_tag`).

## 1. Clean Data

We start by reading the raw CSV file, selecting relevant columns, renaming them, parsing ratings, and generating thumbnails.

In [ ]:
import csv
import os
import re
import uuid
import datetime
import random
import time

# Paths
data_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
input_raw_csv = os.path.join(data_root, 'data', 'raw', 'online-courses', 'Online_Courses.csv')
cleaned_csv = os.path.join(data_root, 'data', 'cleaned', 'cleaned_courses.csv')

# Output Normalized Files
output_courses_csv = os.path.join(data_root, 'data', 'normalized', 'courses.csv')
output_categories_csv = os.path.join(data_root, 'data', 'normalized', 'categories.csv')
output_tags_csv = os.path.join(data_root, 'data', 'normalized', 'tags.csv')
output_course_tags_csv = os.path.join(data_root, 'data', 'normalized', 'course_tag.csv')
output_users_csv = os.path.join(data_root, 'data', 'normalized', 'users.csv')

# Ensure directories exist
os.makedirs(os.path.dirname(cleaned_csv), exist_ok=True)
os.makedirs(os.path.dirname(output_courses_csv), exist_ok=True)

print(f"Raw Data: {input_raw_csv}")
print(f"Cleaned Data: {cleaned_csv}")

In [ ]:
def parse_rating(rating_str):
    if not rating_str:
        return ''
    match = re.search(r"(\d+(\.\d+)?)", rating_str)
    if match:
        return match.group(1)
    return ''

def clean_courses(input_file, output_file):
    columns_mapping = {
        'Title': 'title',
        'Short Intro': 'short_description',
        'Category': 'category',
        'Skills': 'tags',
        'Instructors': 'instructors',
        'Rating': 'rating'
    }

    try:
        with open(input_file, 'r', encoding='utf-8') as infile:
            reader = csv.DictReader(infile)
            
            if not reader.fieldnames:
                print("Error: Empty file or no header found.")
                return

            output_headers = list(columns_mapping.values())
            output_headers.append('thumbnail_url')
            
            with open(output_file, 'w', encoding='utf-8', newline='') as outfile:
                writer = csv.DictWriter(outfile, fieldnames=output_headers)
                writer.writeheader()
                
                rows_processed = 0
                for row in reader:
                    cleaned_row = {}
                    for input_col, output_col in columns_mapping.items():
                        val = row.get(input_col, '')
                        if val is None: val = ''
                        
                        if output_col == 'rating':
                            val = parse_rating(val)
                        elif output_col == 'instructors':
                             if val:
                                parts = val.split(',')
                                if parts:
                                    val = parts[0].strip()

                        cleaned_row[output_col] = val
                    
                    # Skip row if any field is empty
                    if not all(cleaned_row.values()):
                        continue

                    cleaned_row['thumbnail_url'] = f"https://picsum.photos/seed/course_{rows_processed+1:04d}/1280/720"
                    
                    writer.writerow(cleaned_row)
                    rows_processed += 1
                    
            print(f"Successfully processed {rows_processed} rows to {output_file}")

    except FileNotFoundError:
        print(f"Error: Input file not found at {input_file}")
    except Exception as e:
        print(f"An error occurred: {e}")

# Run Cleaning Step
clean_courses(input_raw_csv, cleaned_csv)

## 2. Normalize Categories

Extract distinct categories into `categories.csv` and replace category names with IDs in the courses data.

In [ ]:
def normalize_categories(input_file, output_courses_file, output_categories_file):
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')
    categories_map = {}
    courses_data = []

    with open(input_file, 'r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames
        for row in reader:
            courses_data.append(row)
            category_name = row.get('category', '').strip()
            if category_name and category_name not in categories_map:
                categories_map[category_name] = str(uuid.uuid4())

    # Write Categories
    categories_header = ['id', 'created_at', 'deleted', 'updated_at', 'description', 'name']
    with open(output_categories_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=categories_header)
        writer.writeheader()
        for name, cat_id in categories_map.items():
            writer.writerow({
                'id': cat_id, 'created_at': timestamp, 'deleted': 0, 'updated_at': timestamp,
                'description': name, 'name': name
            })
    print(f"Extracted {len(categories_map)} categories.")

    # Write Courses with Category IDs
    new_fieldnames = [f if f != 'category' else 'category_id' for f in fieldnames]
    with open(output_courses_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=new_fieldnames)
        writer.writeheader()
        for row in courses_data:
            cat_id = categories_map.get(row.get('category', '').strip(), '')
            new_row = {k: (cat_id if k == 'category' else v) for k, v in row.items()}
            # Rename key if needed, or handle above loop logic
            final_row = {}
            for k, v in row.items():
                if k == 'category':
                    final_row['category_id'] = cat_id
                else:
                    final_row[k] = v
            writer.writerow(final_row)

# Run Normalize Categories
normalize_categories(cleaned_csv, output_courses_csv, output_categories_csv)

## 3. Normalize Tags

Extract distinct tags, create `tags` table, create many-to-many `course_tag` table, add IDs to courses, and remove the raw `tags` column.

In [ ]:
def normalize_tags(input_courses_file, output_courses_file, output_tags_file, output_course_tags_file):
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')
    tags_map = {}
    course_tags_relations = []
    updated_courses = []

    with open(input_courses_file, 'r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        course_fieldnames = ['id', 'created_at', 'deleted', 'updated_at'] + [f for f in reader.fieldnames if f != 'tags']

        for row in reader:
            course_id = str(uuid.uuid4())
            
            # Process Tags
            tags_str = row.get('tags', '')
            if tags_str:
                tag_list = [t.strip() for t in tags_str.split(',') if t.strip()]
                for tag_name in tag_list:
                    if tag_name not in tags_map:
                        tags_map[tag_name] = str(uuid.uuid4())
                    
                    course_tags_relations.append({
                        'id': str(uuid.uuid4()),
                        'course_id': course_id,
                        'tag_id': tags_map[tag_name]
                    })
            
            # Process Course Row
            new_row = {'id': course_id, 'created_at': timestamp, 'deleted': 0, 'updated_at': timestamp}
            for k, v in row.items():
                if k != 'tags':
                    new_row[k] = v
            updated_courses.append(new_row)

    # Write Tags
    tags_header = ['id', 'created_at', 'deleted', 'updated_at', 'image_url', 'name']
    with open(output_tags_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=tags_header)
        writer.writeheader()
        for name, t_id in tags_map.items():
            writer.writerow({'id': t_id, 'created_at': timestamp, 'deleted': 0, 'updated_at': timestamp, 'image_url': '', 'name': name})
    print(f"Extracted {len(tags_map)} tags.")

    # Write Course Tags
    with open(output_course_tags_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=['id', 'course_id', 'tag_id'])
        writer.writeheader()
        writer.writerows(course_tags_relations)
    print(f"Created {len(course_tags_relations)} course-tag relations.")

    # Write Courses (overwrite)
    with open(output_courses_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=course_fieldnames)
        writer.writeheader()
        writer.writerows(updated_courses)

# Run Normalize Tags. Note: Input is output_courses_csv from previous step.
normalize_tags(output_courses_csv, output_courses_csv, output_tags_csv, output_course_tags_csv)

## 4. Normalize Users (Instructors)

Extract instructors into `users` table, generating random profile data (username, email, etc.), and link them to courses.

In [ ]:
def normalize_users(input_courses_file, output_courses_file, output_users_file):
    positions = [
        "Software Engineer", "Senior Software Engineer", "Tech Lead",
        "Engineering Manager", "Product Owner", "Business Analyst",
        "System Administrator", "Data Engineer", "Solution Architect", "CTO"
    ]
    users_map = {}
    users_data = []
    updated_courses = []

    with open(input_courses_file, 'r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)
        course_fieldnames = [f if f != 'instructors' else 'instructor_id' for f in reader.fieldnames]

        for row in reader:
            instructor_name = row.get('instructors', '').strip()
            user_id = ''
            if instructor_name:
                if instructor_name not in users_map:
                    user_id = str(uuid.uuid4())
                    users_map[instructor_name] = user_id
                    
                    # Helper for username
                    safe_name = "".join(c for c in instructor_name if c.isalnum()).lower()
                    ts_str = str(int(time.time() * 1000))[-6:]
                    username = f"{safe_name}_{ts_str}"
                    position = random.choice(positions)

                    users_data.append({
                        'id': user_id,
                        'description': f"Instructor for courses. Experienced {position}.",
                        'email': f"{username}@example.com",
                        'image': f"https://api.dicebear.com/7.x/avataaars/svg?seed={username}",
                        'name': instructor_name,
                        'password': f"pass_{str(uuid.uuid4())[:8]}",
                        'phone': f"09{random.randint(10000000, 99999999)}",
                        'position': position,
                        'status': "ACTIVE",
                        'username': username
                    })
                else:
                    user_id = users_map[instructor_name]

            new_row = {}
            for k, v in row.items():
                if k == 'instructors':
                    new_row['instructor_id'] = user_id
                else:
                    new_row[k] = v
            updated_courses.append(new_row)

    # Write Users
    users_header = ['id', 'description', 'email', 'image', 'name', 'password', 'phone', 'position', 'status', 'username']
    with open(output_users_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=users_header)
        writer.writeheader()
        writer.writerows(users_data)
    print(f"Extracted {len(users_data)} instructors/users.")

    # Write Courses (overwrite)
    with open(output_courses_file, 'w', encoding='utf-8', newline='') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=course_fieldnames)
        writer.writeheader()
        writer.writerows(updated_courses)

# Run Normalize Users
normalize_users(output_courses_csv, output_courses_csv, output_users_csv)